# Linear Classification

**Lab 4 · Linear classification with PyTorch**

This lab follows Lab 3 on linear regression. Work in **Google Colab**. No GPU is needed. This notebook creates its own data and does not need variables from the earlier notebooks.

Work from top to bottom. Run each setup cell as supplied, then complete and run each exercise before continuing. Replace every `...` in a starter cell before running it. Try each question in its code cell before opening **Show answer**. To use a supplied answer, copy its code into the corresponding code cell and run it. Opening an answer box does not execute its code.

If a later section depends on code from an earlier one and you are stuck, copy in the supplied answer and continue. The goal is that you can complete the whole lab independently.

We will use the following dataset throughout the lab. Our goal is to use two input features, $x_1$ and $x_2$, to classify the observations into class 0 or class 1.

To match the lecture notation, `inputs` will contain the two observed features, while `X` will denote the **design matrix** obtained by adding a first column of ones. This allows the intercept and the two feature weights to be fitted together.

### How the data are generated

We generate **100 observations: 50 from each class**. The two coordinates are sampled independently within each class from normal (Gaussian) distributions:

| Class | Mean of $x_1$ | Mean of $x_2$ | Standard deviation of $x_1$ | Standard deviation of $x_2$ |
|---|---:|---:|---:|---:|
| 0 | -0.5 | 0.0 | 1.5 | 0.5 |
| 1 | 0.5 | 1.0 | 1.5 | 0.5 |

These are the means and standard deviations of the generating distributions. The finite samples will not have exactly these values. The clouds are more spread out horizontally than vertically, and they overlap.

`t.randn(N // 2, 2)` supplies independent standard normal draws. Multiplying by `spread` sets the two standard deviations; adding the class centre translates the cloud. These two-entry tensors apply coordinate by coordinate to every observation. `t.manual_seed(1)` makes the random draws repeat when this cell is rerun in the same setup. The notebook generates its own data, so no CSV download is required.


### Reading the data plot

The supplied scatter plot uses the two raw input features. `inputs[:, 0]` selects the first feature and `inputs[:, 1]` the second. The one-dimensional list of observed class labels is `y[:, 0]`, because the stored label tensor has shape `(N, 1)`.

We draw the two classes separately, using the already defined `x_class_0` and `x_class_1`. Blue circles represent class 0 and orange triangles represent class 1. The legend identifies the two classes. In each `plt.scatter` call, `x=` and `y=` specify the horizontal and vertical coordinates. The separate tensor named `y` contains the observed class labels: the class is shown by **colour and marker, not height**.

The matrix `X` used for fitting has one extra first column containing ones. Its rows are therefore

$$
(1,\mathbf{x}_i),
$$

exactly as in the lecture. The added `1` is not another observed feature. It is there so that the same matrix product also includes the intercept.

Run the supplied plotting code. You do not need to reconstruct it.


In [ ]:
import torch as t
import matplotlib.pyplot as plt

t.manual_seed(1)  # Repeat the random draws when rerunning this cell in the same setup.

N = 100
# Generate 50 observations for each class, with two input features per observation.
# Each entry of spread is a standard deviation, not a variance.
spread = t.tensor([1.5, 0.5])
x_class_0 = t.randn(N // 2, 2) * spread + t.tensor([-0.5, 0.0])
x_class_1 = t.randn(N // 2, 2) * spread + t.tensor([ 0.5, 1.0])
inputs = t.cat([x_class_0, x_class_1], 0)  # Raw input features; shape (N, 2).

# Labels are in the same order as the observations: 50 zeros, then 50 ones.
y = t.cat([t.zeros(N // 2, 1), t.ones(N // 2, 1)], 0)  # Shape (N, 1).

# Add the constant feature 1 so that the intercept is part of the parameter vector.
X = t.cat([t.ones(N, 1), inputs], dim=1)  # Design matrix; shape (N, 3).

# Horizontal and vertical positions are the two raw input features.
# Draw each observed class with its own colour and marker.
plt.scatter(x=x_class_0[:, 0], y=x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x=x_class_1[:, 0], y=x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.legend()
plt.show()


## 1) Numerical stability in computing the loss

For observation $i$, we use the affine score from the lecture,

$$
f_{\boldsymbol{\theta}}(\mathbf{x}_i)
=
b+\mathbf{x}_i\mathbf{w},
\qquad
\boldsymbol{\theta}
=
\begin{pmatrix}
b\\
\mathbf{w}
\end{pmatrix}.
$$

Here there are two input features, so

$$
\boldsymbol{\theta}
=
\begin{pmatrix}
b\\
w_1\\
w_2
\end{pmatrix}.
$$

Because the first column of the design matrix `X` contains ones, all scores are computed at once as

$$
\mathbf{s}=\mathbf{X}\boldsymbol{\theta}.
$$

The **sigmoid** function $\sigma(s)=1/(1+e^{-s})$ converts each score into the predicted probability of class 1. The probability of class 0 is $1-\sigma(s)=\sigma(-s)$.

In the lecture, the log-loss for a single observation takes a predicted probability $p$ and an observed label $y$:

$$
\ell_{\mathrm{log}}(p,y)
=
-y\log p-(1-y)\log(1-p).
$$

For observation $i$, we use $p_i=\sigma(s_i)$ and the label $y_i$. We therefore evaluate this log-loss for each observation and add the contributions to obtain the empirical logistic loss

$$
\mathcal{L}_{\mathrm{log}}(\boldsymbol{\theta})
=
\sum_{i=1}^N
\ell_{\mathrm{log}}\!\left(\sigma(s_i),y_i\right)
=
-\sum_{i=1}^N
\left[
y_i\log\sigma(s_i)
+
(1-y_i)\log\sigma(-s_i)
\right].
$$

Thus a label $y_i=1$ selects the first term and a label $y_i=0$ selects the second. We use a **sum**, not a mean, as in the lecture.

The Python loss functions we write below will take `theta` as their argument. They will compute the scores from `X @ theta` and use the observed labels stored in `y` to evaluate this sum. In this notebook, the functions will use the data tensors `X` and `y` defined in the setup cell, so we do not pass them as separate arguments. The returned value will therefore be the complete objective $\mathcal{L}_{\mathrm{log}}(\boldsymbol{\theta})$ for the chosen parameters.

### 1.1) Compute the loss using the sigmoid

Write a function `loss_sigmoid(theta)` that takes the full parameter vector and returns the loss. Implement the formula directly using `t.sigmoid` and `t.log`.

**Shapes:** `theta` has shape `(3, 1)`, because it contains the intercept followed by two feature weights. Therefore `X @ theta` and `y` both have shape `(N, 1)`. Keep these column shapes throughout this lab. Mixing `(N,)` and `(N, 1)` would broadcast to `(N, N)` rather than pair each prediction with its own label.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.1</summary>

```python
def loss_sigmoid(theta):
    scores = X @ theta  # One score per observation; shape (N, 1).
    # Sigmoid and log act element by element. .sum() adds all N contributions.
    return -(y * t.log(t.sigmoid(scores))
             + (1 - y) * t.log(t.sigmoid(-scores))).sum()
```

</details>


### 1.2) Numerical instability in the naive loss

Test your function with

$$
\boldsymbol{\theta}
=
\begin{pmatrix}
20\\
20\\
20
\end{pmatrix}.
$$

With the default floating-point precision used in this notebook, you should obtain `nan`, meaning **not a number**, rather than a finite loss. This is the numerical problem we want to investigate. It is not a Python exception.

If you obtain a finite value, check whether you are using a different precision before concluding that your implementation is incorrect.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.2</summary>

```python
theta = 20 * t.ones(3, 1)
print(loss_sigmoid(theta))
```

</details>


### 1.3) Plotting the naive log-sigmoid calculation

Plot $\log\sigma(s)$ for $s$ between $-100$ and $100$, computing it as `t.log(t.sigmoid(s))`.

Also inspect the computed values near $-100$. The visible part of the curve can look reasonable even when some values are `-inf` (negative infinity): those values cannot be displayed as ordinary points on the graph.

**How to draw it:** once you have the equally spaced scores `s` and their computed values `log_p`, use `plt.plot(s, log_p)`, `plt.xlabel("score s")`, `plt.ylabel("log(sigmoid(s))")`, and `plt.show()`. Both tensors are one-dimensional and have the same length. No NumPy conversion is needed here. The values printed below the graph are important: missing infinite values do not become finite just because the visible line looks reasonable.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.3</summary>

```python
s = t.linspace(-100, 100, 100)
log_p = t.log(t.sigmoid(s))

plt.plot(s, log_p)
plt.xlabel('score s')
plt.ylabel('log(sigmoid(s))')
plt.show()

# Look at the calculated values, not only the visible part of the curve.
print(log_p[:5])
```

</details>


### 1.4) Diagnose the problem

Work out why computing $\log\sigma(s)$ in the two separate steps `t.sigmoid(s)` and then `t.log(...)` gives numerical problems for scores $s$ near $-100$.

The key question is: **what floating-point value is actually being passed into the logarithm?**


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.4</summary>

```python
# In this example, the computed sigmoid becomes exactly zero at -100.
print(t.sigmoid(t.tensor(-50.)))
print(t.sigmoid(t.tensor(-100.)))

# Taking the log of that computed zero gives -inf.
print(t.log(t.tensor(0.)))

# A zero label coefficient times -inf gives nan in floating-point arithmetic.
print(t.tensor(0.) * t.log(t.tensor(0.)))
```

Mathematically, $\sigma(-100)$ is positive and its logarithm is finite. The numerical calculation has lost that small positive value **before** taking the logarithm.

This also explains the `nan` in Question 1.2: some computed log-probabilities become `-inf`, and the loss multiplies them by zero label coefficients (`y` or `1-y`). Numerically, `0 * (-inf)` gives `nan`; adding the contributions does not remove it.

</details>


### 1.5) Use the numerically stable `logsigmoid` function

PyTorch provides [`t.nn.functional.logsigmoid`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.logsigmoid.html). It computes the same mathematical quantity $\log\sigma(s)$, but avoids losing the result through the intermediate sigmoid value.

Check this function at $-100$ and $-1000$. Then implement a new loss function, named `loss(theta)`, using `logsigmoid`, and check that it returns a finite value for

$$
\boldsymbol{\theta}
=
\begin{pmatrix}
20\\
20\\
20
\end{pmatrix}.
$$

Pass the **scores** to `logsigmoid`, not their sigmoid values. Use this new `loss` function for the rest of the notebook.

We have **not** changed the mathematical loss. We have only changed how it is evaluated numerically.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.5</summary>

```python
print(t.nn.functional.logsigmoid(t.tensor(-100.)))
print(t.nn.functional.logsigmoid(t.tensor(-1000.)))

def loss(theta):
    scores = X @ theta
    # Same logistic loss as before, now computing each log-probability directly.
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

print(loss(20 * t.ones(3, 1)))
```

</details>


## 2) Train the classifier and draw the results

We will use the stable `loss(theta)` from Question 1.5. Each short exercise below has its own answer. If you get stuck, copy that answer into the exercise cell, run it, and continue.

### 2.1) Implement the gradient

The formula from the lecture is

$$
\nabla\mathcal{L}_{\mathrm{log}}(\boldsymbol{\theta})
=\mathbf{X}^{\top}\bigl(\boldsymbol{\sigma}(\mathbf{X}\boldsymbol{\theta})-\mathbf{y}\bigr).
$$

Complete `grad(theta)` using this formula. `X.mT` is the transpose of `X`, `@` is matrix multiplication, and `t.sigmoid` acts on each score. You do not need to derive the formula again.

**Check:** the result has shape `(3, 1)`. At zero parameters it is approximately $(0,-34.27,-26.99)^{\top}$. The first entry is zero because there are equally many observations in each class and every predicted probability is initially 0.5.

In [ ]:
def grad(theta):
    scores = ...
    probabilities = ...
    return ...

print(grad(t.zeros(3, 1)))


<details>
<summary>Show answer — 2.1</summary>

```python
def grad(theta):
    scores = X @ theta
    probabilities = t.sigmoid(scores)
    return X.mT @ (probabilities - y)

print(grad(t.zeros(3, 1)))
```



</details>

### 2.2) Count classification mistakes

Write a function `training_mistakes(theta)` that returns the number of incorrectly classified observations.

1. Compute the probabilities with `t.sigmoid(X @ theta)`.
2. Apply the plug-in rule: predict class 1 when the probability is **greater than** `0.5`; otherwise predict class 0, including a tie.
3. Compare the predicted labels with `y` and count the differences.

**Python help:** `probabilities > 0.5` gives Boolean values. `.to(y.dtype)` writes them as numeric zeros and ones. `!=` means “not equal”, `.sum()` counts the `True` entries, and `.item()` extracts the single number from the resulting tensor.

**Check:** `training_mistakes(t.zeros(3, 1))` should return `50`: all probabilities are 0.5, so all predictions are class 0.

In [ ]:
def training_mistakes(theta):
    probabilities = ...
    predicted_labels = ...
    return ...

print(training_mistakes(t.zeros(3, 1)))


<details>
<summary>Show answer — 2.2</summary>

```python
def training_mistakes(theta):
    probabilities = t.sigmoid(X @ theta)
    predicted_labels = (probabilities > 0.5).to(y.dtype)
    return (predicted_labels != y).sum().item()

print(training_mistakes(t.zeros(3, 1)))
```



</details>

### 2.3) Run gradient descent

Start from zero parameters and make **100 updates**, using learning rate `0.01`:

$$
\boldsymbol{\theta}^{(t+1)}
=\boldsymbol{\theta}^{(t)}-0.01\,\nabla\mathcal{L}_{\mathrm{log}}(\boldsymbol{\theta}^{(t)}).
$$

Complete the **three marked lines** in the cell below: update `theta`, record its loss, and record its mistake count. Use `grad(theta)`, `loss(theta).item()`, and `training_mistakes(theta)`.

`.append(value)` adds a value to the end of a list. Entry 0 already records the initial model. Append the new values **after** each update, so entry `step` corresponds to exactly `step` completed updates.

The supplied `if` statements keep copies of the coefficients after 1, 10, and 100 updates. `.clone()` makes a separate copy. Leave these lines as they are; we will use the saved coefficients to draw the lines later.

Rerunning this whole cell restarts training from zero. It does not continue the previous run.

In [ ]:
learning_rate = 0.01
theta = t.zeros(3, 1)
loss_history = [loss(theta).item()]
mistake_history = [training_mistakes(theta)]

for step in range(1, 101):
    theta = ...  # Complete the update.
    loss_history.append(...)  # Record the loss after this update.
    mistake_history.append(...)  # Record the mistake count.

    # Supplied: keep copies for the later plotting exercise.
    if step == 1:
        theta_1 = theta.clone()
    if step == 10:
        theta_10 = theta.clone()
    if step == 100:
        theta_100 = theta.clone()

print("Recorded values:", len(loss_history), len(mistake_history))
print("Final loss:", loss_history[-1])
print("Final mistakes:", mistake_history[-1])


<details>
<summary>Show answer — 2.3</summary>

```python
learning_rate = 0.01
theta = t.zeros(3, 1)
loss_history = [loss(theta).item()]
mistake_history = [training_mistakes(theta)]

for step in range(1, 101):
    theta = theta - learning_rate * grad(theta)
    loss_history.append(loss(theta).item())
    mistake_history.append(training_mistakes(theta))

    # Supplied: keep copies for the later plotting exercise.
    if step == 1:
        theta_1 = theta.clone()
    if step == 10:
        theta_10 = theta.clone()
    if step == 100:
        theta_100 = theta.clone()

print("Recorded values:", len(loss_history), len(mistake_history))
print("Final loss:", loss_history[-1])
print("Final mistakes:", mistake_history[-1])
```

There should be **101 entries** in each list: the initial model plus 100 updated models. You can check individual entries, for example `loss_history[10]` and `mistake_history[10]`.

| Completed updates | Summed logistic loss | Training mistakes |
|---:|---:|---:|
| 0 | 69.31 | 50 |
| 1 | 55.86 | 21 |
| 10 | 38.01 | 15 |
| 100 | 25.65 | 9 |

The initial loss is $100\log 2$, because all probabilities are 0.5.

</details>

### 2.4) Check the fitted model and the mistake bound

Print the fitted coefficients `theta`, the number of training mistakes, the loss, and the loss divided by $\log 2$.

If $\mathcal{E}(\boldsymbol{\theta})$ denotes the **number** of training mistakes, the bound from the lecture is

$$
\mathcal{E}(\boldsymbol{\theta})\leq
\frac{\mathcal{L}_{\mathrm{log}}(\boldsymbol{\theta})}{\log 2}.
$$

The first line below calculates the natural logarithm of 2. Complete the remaining lines using the two functions you have written.

In [ ]:
log_two = t.log(t.tensor(2.0)).item()

print("Coefficients:", theta)
print("Number of mistakes:", ...)
print("Summed logistic loss:", ...)
print("Upper bound on mistakes:", ...)


<details>
<summary>Show answer — 2.4</summary>

```python
log_two = t.log(t.tensor(2.0)).item()

print("Coefficients:", theta)
print("Number of mistakes:", training_mistakes(theta))
print("Summed logistic loss:", loss(theta).item())
print("Upper bound on mistakes:", loss(theta).item() / log_two)
```

The coefficients should be approximately $(-1.5023,\,0.6202,\,4.0354)^{\top}$, in the order $(b,w_1,w_2)^{\top}$.

There are **9 mistakes**, a summed loss of about **25.65**, and a bound of about **37.01**. The bound can be larger than the actual mistake count. These are the results after 100 updates; we are not claiming convergence or the minimum possible mistake count.

</details>

### 2.5) Draw the decision boundaries

The decision boundary is the straight line

$$b+w_1x_1+w_2x_2=0.$$

Rearrange this equation to express $x_2$ in terms of $x_1$. For the three saved models, $w_2$ is nonzero, so division by $w_2$ is valid.

Draw the observations and the three lines after **1, 10, and 100 updates on the same plot**. Two endpoints are enough to draw each straight line. Keep the observations fixed and use a legend to identify the three lines.

First run the supplied setup cell below. It gives all three lines the same two horizontal endpoints and fixes axis limits that include every observation. This also stops a steep line from stretching the vertical axis.

In [ ]:
# Supplied plotting limits: use these unchanged for all three boundaries.
x1_min = inputs[:, 0].min().item() - 0.5
x1_max = inputs[:, 0].max().item() + 0.5
x2_min = inputs[:, 1].min().item() - 0.5
x2_max = inputs[:, 1].max().item() + 0.5
x1_line = t.tensor([x1_min, x1_max])


Complete the three missing expressions below and then write the plotting commands.

`b, w1, w2 = theta_1[:, 0]` reads the three coefficients from the saved column. The same line with `theta_10` or `theta_100` reads the later coefficients.

**Plotting help:**

- Start a fresh plot with `plt.figure()`.
- Copy the two `plt.scatter` calls from the initial data plot. The blue circles and orange triangles show the observed classes, not the predictions.
- Use `plt.plot(x1_line, x2_after_1, color='black', linestyle=':', label='After 1 update')` for the first line. Add two more `plt.plot` calls, using `linestyle='--'` after 10 updates and `linestyle='-'` after 100 updates.
- Use `plt.xlim(x1_min, x1_max)` and `plt.ylim(x2_min, x2_max)` to keep the supplied limits.
- Add axis labels. Use `plt.legend(loc='upper left', bbox_to_anchor=(1, 1))` to put the legend beside the plot so that it does not cover observations, then `plt.tight_layout()` and `plt.show()`.

Here all three saved values of $w_2$ are positive: each model predicts class 1 **above its own line**, and class 0 below it or on it. All-zero coefficients at step 0 do not define a single separating line, so we start the comparison at step 1.

In [ ]:
b, w1, w2 = theta_1[:, 0]
x2_after_1 = ...

b, w1, w2 = theta_10[:, 0]
x2_after_10 = ...

b, w1, w2 = theta_100[:, 0]
x2_after_100 = ...

# Draw the observations and the three lines here.


<details>
<summary>Show answer — 2.5</summary>

```python
b, w1, w2 = theta_1[:, 0]
x2_after_1 = -(b + w1 * x1_line) / w2

b, w1, w2 = theta_10[:, 0]
x2_after_10 = -(b + w1 * x1_line) / w2

b, w1, w2 = theta_100[:, 0]
x2_after_100 = -(b + w1 * x1_line) / w2

plt.figure()
plt.scatter(x=x_class_0[:, 0], y=x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x=x_class_1[:, 0], y=x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
plt.plot(x1_line, x2_after_1, color='black', linestyle=':', label='After 1 update')
plt.plot(x1_line, x2_after_10, color='black', linestyle='--', label='After 10 updates')
plt.plot(x1_line, x2_after_100, color='black', linestyle='-', label='After 100 updates')
plt.xlim(x1_min, x1_max)
plt.ylim(x2_min, x2_max)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()
```

Rearranging gives $x_2=-(b+w_1x_1)/w_2$. The first line leaves 21 mistakes; the later models make 15 and then 9 mistakes. Both the slope and the intercept of the line change.

</details>

### 2.6) Plot the training curves

The two history lists contain the values at updates 0 to 100. You have already used `plt.plot` in Question 1.3; now use it to see what happened during training.

**(a) Plot the summed logistic loss.** Use `steps` for the horizontal coordinates and `loss_history` for the vertical coordinates. Label both axes.

In [ ]:
steps = range(len(loss_history))

# Plot loss_history against steps, label both axes, and display the figure.


<details>
<summary>Show answer — 2.6a</summary>

```python
steps = range(len(loss_history))

plt.figure()
plt.plot(steps, loss_history)
plt.xlabel('Completed updates')
plt.ylabel('Summed logistic loss')
plt.show()
```

The loss decreases from about 69.31 to 25.65 in this run.

</details>

**(b) Compare the mistake count with its upper bound.** On one new plot, draw both the logistic loss divided by $\log 2$ and the number of mistakes. Label the axes and add a legend.

The supplied line below converts the recorded losses into a tensor and divides each value by $\log 2$. It is only a change of scale for this plot: the training updates still use the original summed logistic loss.

Use two `plt.plot` calls, with `steps` as their horizontal coordinates and `scaled_loss_history` and `mistake_history` as their respective vertical coordinates. Set a `label=` in each call and use `plt.legend()`. The lines connect measurements at integer update numbers.

In [ ]:
scaled_loss_history = t.tensor(loss_history) / log_two

# Plot scaled_loss_history and mistake_history against steps on the same axes.
# Add axis labels and a legend, then display the figure.


<details>
<summary>Show answer — 2.6b</summary>

```python
scaled_loss_history = t.tensor(loss_history) / log_two

plt.figure()
plt.plot(steps, scaled_loss_history, label='Logistic loss / log(2)')
plt.plot(steps, mistake_history, label='Number of mistakes')
plt.xlabel('Completed updates')
plt.ylabel('Scaled loss or mistake count')
plt.legend()
plt.show()
```

At update 0 the two values are 100 and 50. At update 100 they are approximately 37.01 and 9. The mistake count lies at or below the scaled loss at every update, as required by the bound.

</details>

### 2.7) What did the updates improve?

1. How does the decision boundary change between updates 1, 10, and 100? What happens to the number of mistakes?
2. Does the mistake count always decrease when the logistic loss decreases? Compare `loss_history[4]` with `loss_history[5]`, and `mistake_history[4]` with `mistake_history[5]`.

<details>
<summary>Show answer — 2.7</summary>

Further updates rotate and translate the line. The number of mistakes falls from **21 after one update** to **15 after 10** and **9 after 100**. The observations stay fixed. The second coordinate is more informative here because it has less spread within each class; its fitted weight becomes larger than that of the first coordinate.

The algorithm minimises logistic loss, not the number of mistakes. Between updates **4 and 5**, the loss falls from about **46.28 to 44.37**, but the mistake count rises from **19 to 20**. A probability can cross the classification threshold and add a mistake while the total loss improves across the dataset. A decrease in total loss does not mean that every observation's individual loss decreases.

The bound has only one direction: small logistic loss guarantees an upper bound on mistakes; few mistakes do not by themselves guarantee small logistic loss. All measurements here are on the training data, not on new observations.

**If you try changing the learning rate:** after the first update from zero, $\boldsymbol{\theta}^{(1)}=-\eta\nabla\mathcal{L}_{\mathrm{log}}(\mathbf{0})$. Changing positive $\eta$ rescales all coefficients, so the first boundary and predicted labels stay the same, while probabilities and loss change. This first-step invariance does not hold in general for later updates.

</details>